
# 📄 Document Question Answering System using Retrieval-Augmented Generation (RAG)

A complete, beginner-friendly, **fully local** RAG pipeline in a single notebook.

Upload PDF documents, ask natural-language questions, and get answers generated
**only** from the retrieved document content — never from the model's own
memory. If the answer isn't in your documents, the system will honestly say:

> "I could not find this information in the uploaded documents."

**Pipeline overview:**

```
PDF files -> Text Extraction -> Chunking -> Embeddings -> FAISS Index
                                                                 |
Question -> Query Embedding -> Similarity Search -> Top-K Chunks
                                                                 |
                                          Retrieved Chunks -> LLM -> Final Answer
```

**Tech stack (100% free, no API keys, runs on CPU):**
| Component | Tool |
|---|---|
| PDF parsing | PyPDF |
| Chunking | LangChain `RecursiveCharacterTextSplitter` |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector store | FAISS |
| Language model | `google/flan-t5-base` (Hugging Face, local) |
| UI | Gradio |

Run the cells in order from top to bottom.



## 0. Install Dependencies

Run this once. If you're in a fresh environment (Colab, venv, etc.) this
installs everything needed — no API keys required.


In [ ]:
# Install all required libraries (uncomment the line below if not already installed)
# %pip install -q pypdf langchain langchain-community langchain-text-splitters langchain-huggingface \
#     sentence-transformers faiss-cpu transformers torch gradio
print("If you haven't already, run the pip install command above (uncomment it) before continuing.")



## 1. Imports & Configuration

All shared settings live here: folder paths, chunk size/overlap, model names,
and how many chunks to retrieve per question.


In [ ]:
import os
import shutil

from pypdf import PdfReader

# The Document class moved from langchain.docstore.document to
# langchain_core.documents in newer LangChain versions.
try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.docstore.document import Document

# RecursiveCharacterTextSplitter moved to its own package in newer LangChain versions.
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

# HuggingFaceEmbeddings moved to a standalone integration package
# (langchain-huggingface) in newer LangChain versions.
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS
from transformers import pipeline as hf_pipeline

import gradio as gr


In [ ]:
# ----------------------------- Configuration -----------------------------

# Folder where uploaded / sample PDFs live
DOCUMENTS_DIR = "documents"

# Folder where the FAISS index is saved to disk
VECTOR_STORE_DIR = "vector_store"

# Name used when saving/loading the FAISS index
FAISS_INDEX_NAME = "faiss_index"

# Embedding model: small, fast, and good enough for beginner projects.
# Produces 384-dimensional vectors and runs comfortably on CPU.
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Local, free, no-API-key language model used to generate answers.
# Swap for "google/flan-t5-large" if you have more RAM/CPU for better answers.
LLM_MODEL_NAME = "google/flan-t5-base"

# Text splitting configuration (explained in Section 3 below)
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

# How many chunks to retrieve per question
TOP_K = 3

# The exact fallback message required when the answer is not in the documents
NOT_FOUND_MESSAGE = "I could not find this information in the uploaded documents."


def ensure_directories_exist():
    # Make sure documents/ and vector_store/ exist.
    os.makedirs(DOCUMENTS_DIR, exist_ok=True)
    os.makedirs(VECTOR_STORE_DIR, exist_ok=True)


def list_pdf_files(folder=DOCUMENTS_DIR):
    # Return full paths to every .pdf file inside `folder`.
    ensure_directories_exist()
    return [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(".pdf")
    ]


def vector_store_exists():
    # Check whether a previously-built FAISS index is saved on disk.
    index_path = os.path.join(VECTOR_STORE_DIR, FAISS_INDEX_NAME)
    return (
        os.path.exists(os.path.join(index_path, "index.faiss"))
        and os.path.exists(os.path.join(index_path, "index.pkl"))
    )


ensure_directories_exist()
print("Config loaded. Documents folder:", os.path.abspath(DOCUMENTS_DIR))



## 2. Step 1 — Document Loading

Read every PDF inside `documents/` and extract its raw text using **PyPDF**,
page by page. Each extracted page becomes a LangChain `Document` with
metadata (`source` filename + `page` number) so answers can later be traced
back to exactly where they came from.

Place your own PDFs (notes, resume, research papers, books) inside the
`documents/` folder before running this cell — a `sample.pdf` about RAG is
included so you can try things out immediately.


In [ ]:
def load_pdfs(folder=DOCUMENTS_DIR):
    # Load every PDF in `folder` and return a list of Document objects.
    ensure_directories_exist()
    pdf_paths = list_pdf_files(folder)

    if not pdf_paths:
        print(f"No PDF files found in '{folder}/'. Add some PDFs and try again.")
        return []

    documents = []

    for pdf_path in pdf_paths:
        filename = os.path.basename(pdf_path)
        print(f"Loading: {filename}")

        try:
            reader = PdfReader(pdf_path)
        except Exception as e:
            print(f"  Could not read {filename}: {e}")
            continue

        for page_number, page in enumerate(reader.pages, start=1):
            text = (page.extract_text() or "").strip()
            if not text:
                continue  # skip blank / image-only pages (no OCR in this beginner project)

            documents.append(
                Document(page_content=text, metadata={"source": filename, "page": page_number})
            )

    print(f"Loaded {len(documents)} pages of text from {len(pdf_paths)} PDF(s).")
    return documents


# Run it
raw_documents = load_pdfs()



## 3. Step 2 — Text Chunking

Large pages of text aren't ideal to feed directly into an embedding model or
an LLM, so they're split into smaller, overlapping chunks using LangChain's
`RecursiveCharacterTextSplitter`.

**Why `CHUNK_SIZE = 500` and `CHUNK_OVERLAP = 100`?**

- **500 characters** (~100–125 words) keeps each chunk focused on a single
  idea, which makes embeddings more accurate — a chunk about *one topic*
  embeds better than a chunk that jumps between five topics. It's also small
  enough that several retrieved chunks comfortably fit inside a small LLM's
  limited context window.
- **100 characters of overlap** (20% of the chunk size) prevents important
  sentences from being awkwardly cut in half at chunk boundaries. Without
  overlap, "The capital of France is Paris." could be split into "The
  capital of France is" + "Paris.", losing the answer in either half.

`RecursiveCharacterTextSplitter` tries natural boundaries first (paragraphs,
then sentences, then words), only falling back to a hard character cut as a
last resort — keeping chunks more readable than a naive fixed-length split.


In [ ]:
def split_documents(documents):
    # Split loaded page-documents into smaller overlapping chunks.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    print(f"Split into {len(chunks)} chunks (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}).")
    return chunks


# Run it
text_chunks = split_documents(raw_documents) if raw_documents else []
if text_chunks:
    print("\nExample chunk:\n", text_chunks[0].page_content[:200])
    print("\nMetadata:", text_chunks[0].metadata)



## 4. Step 3 & 4 — Embedding Creation + FAISS Storage

Each text chunk is converted into a numerical vector (embedding) using
`all-MiniLM-L6-v2`, then all vectors are stored in a **FAISS** index so we
can later search for the chunks most similar to a user's question.

The index is saved to disk (`vector_store/`) so it doesn't need to be
rebuilt every time you restart the notebook — see the "reload" cell below.


In [ ]:
def build_vector_store(chunks):
    # Embed chunks with MiniLM and store them in a FAISS index saved to disk.
    print("Loading embedding model (all-MiniLM-L6-v2)... this may take a moment.")
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

    print("Creating embeddings and building FAISS index...")
    vector_store = FAISS.from_documents(chunks, embeddings)

    ensure_directories_exist()
    save_path = os.path.join(VECTOR_STORE_DIR, FAISS_INDEX_NAME)
    vector_store.save_local(save_path)
    print(f"FAISS index saved to: {save_path}")

    return embeddings, vector_store


def load_existing_vector_store():
    # Reload a previously saved FAISS index from disk without rebuilding it.
    if not vector_store_exists():
        print("No saved FAISS index found.")
        return None, None

    print("Loading embedding model (all-MiniLM-L6-v2)...")
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

    print("Loading existing FAISS index from disk...")
    index_path = os.path.join(VECTOR_STORE_DIR, FAISS_INDEX_NAME)
    vector_store = FAISS.load_local(
        index_path, embeddings, allow_dangerous_deserialization=True
    )
    return embeddings, vector_store


def ingest_all():
    # Full pipeline: load -> split -> embed -> store. Returns (embeddings, vector_store).
    documents = load_pdfs()
    if not documents:
        return None, None
    chunks = split_documents(documents)
    return build_vector_store(chunks)


In [ ]:
# Build the index (or reload it if one already exists on disk)
if vector_store_exists():
    embeddings, vector_store = load_existing_vector_store()
else:
    embeddings, vector_store = ingest_all()

print("\nVector store ready!" if vector_store is not None else "\nNo vector store yet — add PDFs to documents/ and re-run.")



## 5. Load the Language Model

`google/flan-t5-base` is a free, local Hugging Face text2text model — no API
key, no internet call at inference time (only the first download needs
internet). Swap in `google/flan-t5-large` if you want slightly better
answers and have more RAM/CPU.


In [ ]:
print(f"Loading language model ({LLM_MODEL_NAME})... this may take a moment.")
llm = hf_pipeline("text2text-generation", model=LLM_MODEL_NAME, max_new_tokens=200)
print("Language model ready!")



## 6. Steps 5–10 — Query → Retrieval → Grounded Answer

Given a question, this section:

1. Embeds the question with the **same** embedding model used for the documents
2. Searches FAISS for the **Top-3** most similar chunks (similarity search)
3. Displays those retrieved chunks (so you can see exactly how RAG works)
4. Passes **only** that retrieved context to the language model
5. Returns the exact fallback message if the answer isn't present in the context — never hallucinating


In [ ]:
def retrieve(question, k=TOP_K):
    # Embed the question, search FAISS, and return (Document, score) pairs.
    # Lower FAISS L2 score = more similar.
    if vector_store is None:
        raise ValueError("Vector store is not loaded. Ingest documents first (see Section 4).")
    return vector_store.similarity_search_with_score(question, k=k)


def format_context(retrieved_chunks):
    # Pretty-print retrieved chunks with source, page, and similarity score.
    if not retrieved_chunks:
        return "No context retrieved."
    lines = []
    for i, (doc, score) in enumerate(retrieved_chunks, start=1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        lines.append(
            f"--- Chunk {i} (source: {source}, page {page}, distance score: {score:.4f}) ---\n"
            f"{doc.page_content}\n"
        )
    return "\n".join(lines)


def generate_answer(question, retrieved_chunks):
    # Ask the LLM to answer using ONLY the retrieved context.
    context_text = "\n\n".join(doc.page_content for doc, _score in retrieved_chunks)

    if not context_text.strip():
        return NOT_FOUND_MESSAGE

    prompt = (
        "You are a helpful assistant that answers questions using ONLY the "
        "context below. Do not use any outside knowledge.\n"
        "If the answer cannot be found in the context, reply exactly with: "
        f'"{NOT_FOUND_MESSAGE}"\n\n'
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    result = llm(prompt)
    return result[0]["generated_text"].strip()


def answer_question(question, k=TOP_K):
    # Run retrieval + generation together. Returns (answer, retrieved_chunks).
    retrieved_chunks = retrieve(question, k=k)
    answer_text = generate_answer(question, retrieved_chunks)
    return answer_text, retrieved_chunks



## 7. Try It Out

Ask a question about whatever PDFs are in `documents/`. With the included
`sample.pdf` (notes about RAG itself), try something like:
*"What is RAG?"* or *"What are the main components of a RAG system?"*


In [ ]:
question = "What is RAG?"  # <-- change this to ask your own question

answer_text, retrieved_chunks = answer_question(question)

print("QUESTION:\n", question)
print("\nRETRIEVED CONTEXT:\n")
print(format_context(retrieved_chunks))
print("\nFINAL ANSWER:\n", answer_text)


In [ ]:
# Example of a question NOT covered by the documents -> should trigger the fallback message
question2 = "What is the capital of Japan?"

answer_text2, retrieved_chunks2 = answer_question(question2)

print("QUESTION:\n", question2)
print("\nFINAL ANSWER:\n", answer_text2)



## 8. Add More PDFs (Optional)

Copy any new PDF into `documents/`, then re-run this cell to rebuild the
index so it includes the new file(s). Supports multiple PDFs at once.


In [ ]:
def add_pdfs_and_reindex(pdf_paths):
    # Copy the given PDF file paths into documents/ and rebuild the FAISS index.
    # Example: add_pdfs_and_reindex(["/path/to/my_resume.pdf"])
    global embeddings, vector_store

    ensure_directories_exist()
    for path in pdf_paths:
        shutil.copy(path, os.path.join(DOCUMENTS_DIR, os.path.basename(path)))
        print(f"Copied: {os.path.basename(path)}")

    embeddings, vector_store = ingest_all()
    print("Re-indexing complete!" if vector_store is not None else "No readable PDFs found.")


# Example usage (uncomment and edit the path):
# add_pdfs_and_reindex(["/path/to/your_document.pdf"])



## 9. Interactive Gradio Interface

A simple UI with the four required panels: **Upload PDF**, **Ask Question**,
**Retrieved Context**, and **Final Answer**. Run the cell below and either
open the local URL it prints, or use it inline in the notebook.


In [ ]:
def handle_upload(uploaded_files):
    # Gradio callback: copy uploaded PDFs into documents/ and re-index.
    global embeddings, vector_store

    if not uploaded_files:
        return "No file selected."
    if not isinstance(uploaded_files, list):
        uploaded_files = [uploaded_files]

    ensure_directories_exist()
    saved_names = []
    for file_obj in uploaded_files:
        src_path = file_obj.name
        filename = os.path.basename(src_path)
        shutil.copy(src_path, os.path.join(DOCUMENTS_DIR, filename))
        saved_names.append(filename)

    print(f"Uploaded: {', '.join(saved_names)}. Indexing now...")
    embeddings, vector_store = ingest_all()

    if vector_store is None:
        return "Upload succeeded but no readable text was found in the PDF(s)."
    return f"Indexed {len(saved_names)} file(s): {', '.join(saved_names)}. Ready to answer questions!"


def handle_question(question):
    # Gradio callback: run retrieval + generation for the UI.
    if not question or not question.strip():
        return "Please enter a question.", ""
    if vector_store is None:
        return "No documents indexed yet. Please upload a PDF first.", ""
    try:
        answer_text, retrieved_chunks = answer_question(question)
    except Exception as e:
        return f"Error during retrieval: {e}", ""
    return format_context(retrieved_chunks), answer_text


with gr.Blocks(title="Document QA using RAG") as demo:
    gr.Markdown("# 📄 Document Question Answering System (RAG)")
    gr.Markdown(
        "Upload PDF documents, then ask questions about them. Answers are generated "
        "**only** from the retrieved document content."
    )

    status_box = gr.Textbox(
        label="Status",
        value="Ready to answer questions." if vector_store is not None else "No documents indexed yet. Upload a PDF to get started.",
        interactive=False,
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Upload PDF")
            file_upload = gr.File(label="Upload one or more PDF files", file_types=[".pdf"], file_count="multiple")
            upload_button = gr.Button("Index Uploaded PDF(s)")
        with gr.Column(scale=1):
            gr.Markdown("### 2. Ask a Question")
            question_box = gr.Textbox(label="Your question", placeholder="e.g. What is the main topic of this document?")
            ask_button = gr.Button("Ask", variant="primary")

    gr.Markdown("### 3. Retrieved Context")
    context_output = gr.Markdown(value="Ask a question to see the retrieved chunks here.")

    gr.Markdown("### 4. Final Answer")
    answer_output = gr.Textbox(label="Answer", interactive=False)

    upload_button.click(fn=handle_upload, inputs=file_upload, outputs=status_box)
    ask_button.click(fn=handle_question, inputs=question_box, outputs=[context_output, answer_output])
    question_box.submit(fn=handle_question, inputs=question_box, outputs=[context_output, answer_output])

demo.launch(inline=True, share=False)



## 10. Summary

You've built a complete, local RAG pipeline:

1. **Loaded** PDFs with PyPDF
2. **Chunked** text (500 chars, 100 overlap) with `RecursiveCharacterTextSplitter`
3. **Embedded** chunks with `all-MiniLM-L6-v2`
4. **Stored** vectors in FAISS (saved to `vector_store/` for instant reloads)
5. **Retrieved** the Top-3 most relevant chunks per question
6. **Generated** grounded answers with `google/flan-t5-base`, with an honest fallback when the answer isn't present

**Ideas to extend this project:**
- Add OCR for scanned/image-based PDFs
- Try `google/flan-t5-large` for higher-quality answers
- Highlight the exact sentence within a chunk that answered the question
- Add multi-turn conversational memory
